## Predictive Maintenance & Equitable Water Infrastructure Using Data-Driven Technology
*Strategic focus on SDG 6: Clean Water and Sanitation*

## Elevator Pitch
In rural Kenya, water infrastructure failure is a silent crisis that disrupts thousands of lives daily. Using the **Kenya Water Points Dataset (WPdx)**, our team developed a model designed to identify at-risk water points before they fail. By moving from reactive repairs to a data-driven predictive roadmap, this model provides government agencies and NGOs with the insights needed for proactive maintenance, ensuring consistent and reliable water access for rural communities.

## The Project Team
* **Abdullahi Abdi Hassan (Statistics):** Responsible for statistical validation, baseline modeling, and defining mathematical thresholds for evaluation metrics (Recall and F1-Score).
* **Claire (Data Architect):** Responsible for high-volume data ingestion and intensive initial cleaning of the messy, real-world CSV dataset.
* **Yvonne (Feature Engineer):** Responsible for creating domain-specific features, such as infrastructure age and standardized management categories.
* **Dahir (ML Engineer):** Responsible for implementation of advanced ensemble algorithms (Random Forest/XGBoost) and hyperparameter optimization.
* **Lauren (Visualization Specialist):** Responsible for designing high-impact technical charts and geospatial risk hotspot visualizations.
* **Samantha (Pipeline Engineer):** Responsible for developing the end-to-end automated Python pipeline.

---

# 1. Business Understanding

## 1.1 The Problem
Sustainable access to clean water is a fundamental human right. However, maintenance in Kenya has historically been **reactive**—repairs only occur after a pump has already failed. Research indicates that approximately **30% of rural water points in sub-Saharan Africa are non-functional** at any given time. This leads to "sunk costs" for donors and, more importantly, a loss of basic services for communities.

## 1.2 Stakeholders
* **Ministry of Water, Sanitation & Irrigation:** National policy-making for water security.
* **County Governments:** Strategic optimization of maintenance budgets and field repair targeting.
* **WPdx & NGOs (e.g., Evidence Action):** Improving data-driven field operations.
* **UNICEF & Local Communities:** Beneficiaries of reliable water and improved health outcomes.

## 1.3 Modeling Strategy: The Statistical Baseline Approach
We have implemented a tiered modeling strategy to ensure the technical solution is robust and adds measurable value:
* **Interpretability Baseline (Logistic Regression):** We first establish a performance benchmark using a simple Logistic Regression. This allows us to quantify the linear relationship between features (like pump age) and failure, providing a "status quo" reference point.
* **Advanced Ensembles (Random Forest/XGBoost):** We then deploy complex non-linear models. We justify their use only if they demonstrate a significant "performance premium" (higher Recall and ROC-AUC) over the baseline.

## 1.4 Success Criteria
* **Statistical Target:** Achieve a **Primary Metric of ROC-AUC $\ge$ 0.80**. Critically, the chosen model must significantly outperform the **Logistic Regression baseline** to justify the complexity.
* **Business Target:** **Maximize Recall for the "Non-Functional" class**. Our priority is to minimize "False Negatives" (failing to identify a pump that is about to break) to ensure no community is left without service.

## 1.5 Impact & Rationale
This project transforms raw data into a **Decision Support System**. By shifting the focus to predictive indicators, stakeholders can intervene before a breakdown occurs, significantly reducing "water poverty" periods and optimizing the allocation of limited repair budgets.

## 1.6 Data Source
**Source:** [WPdx Kenya Dataset](https://data.waterpointdata.org/dataset/Kenya-Data/e2gs-xfxf)  
**Scope:** 21,300+ entries covering diverse technologies and regional patterns across Kenya.

# 2. Data Understanding
In this phase, we load the data and handle the technical metadata (HXL tags) that comes with international water datasets.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Load data - Skipping the HXL metadata row (row index 1)
df = pd.read_csv('wpdx_enhanced.csv', skiprows=[1])
df.head() # checking the first 5 rows

,lat_deg,lon_deg,status_id,report_date,source,water_source_clean,water_tech_clean,clean_country_id,clean_country_name,clean_adm1,...,is_urban,days_since_report,staleness,prediction_yes_0y,prediction_yes_2y,prediction_no_0y,prediction_no_2y,predicted_status_0y,predicted_status_2y,predicted_category
0,0.212005,34.615833,Yes,2023-10-30,Evidence Action,Protected Well,Motorized Pump - Electric,KEN,Kenya,Kakamega,...,False,769,78.474305,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.212567,34.616543,Yes,2023-10-30,Evidence Action,Protected Well,Motorized Pump - Electric,KEN,Kenya,Kakamega,...,False,769,78.474305,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.222063,34.601463,Yes,2023-11-16,Evidence Action,Protected Well,Motorized Pump - Electric,KEN,Kenya,Kakamega,...,False,752,78.895948,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.222362,34.604015,Yes,2023-11-18,Evidence Action,Protected Well,Motorized Pump - Electric,KEN,Kenya,Kakamega,...,False,750,78.945701,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.023602,34.777866,Yes,2023-09-11,Evidence Action,Protected Well,Motorized Pump - Electric,KEN,Kenya,Vihiga,...,False,818,77.271546,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
# Initial Statistical Audit
print(f"Total Records: {df.shape[0]}")
print(f"Total Features: {df.shape[1]}")
print("\nMissing values in key columns:")
print(df[['status_clean', 'install_year', 'water_source_clean']].isnull().sum()) # missing values

Total Records: 21953
Total Features: 54

Missing values in key columns:
status_clean              0
install_year          11787
water_source_clean      128
dtype: int64


In [3]:
# Checking class distribution for imbalanced learning strategy
print("\nTarget Class Distribution:")
print(df['status_clean'].value_counts(normalize=True))


Target Class Distribution:
status_clean
Non-Functional                0.618503
Functional, not in use        0.216690
Functional                    0.089464
Functional, needs repair      0.066551
Non-Functional, dry season    0.008655
Abandoned/Decommissioned      0.000137
Name: proportion, dtype: float64


In [4]:
df.info()  # summary of rows and columns and missing values

<class 'pandas.DataFrame'>
RangeIndex: 21953 entries, 0 to 21952
Data columns (total 54 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   lat_deg                  21953 non-null  float64
 1   lon_deg                  21953 non-null  float64
 2   status_id                21953 non-null  str    
 3   report_date              21953 non-null  str    
 4   source                   21953 non-null  str    
 5   water_source_clean       21825 non-null  str    
 6   water_tech_clean         16034 non-null  str    
 7   clean_country_id         21953 non-null  str    
 8   clean_country_name       21953 non-null  str    
 9   clean_adm1               21953 non-null  str    
 10  clean_adm2               21953 non-null  str    
 11  clean_adm3               21953 non-null  str    
 12  clean_adm4               0 non-null      float64
 13  activity_id              19744 non-null  str    
 14  scheme_id                0 non-nu

In [5]:
df.duplicated() # checking duplicates

0        False
1        False
2        False
3        False
4        False
         ...  
21948    False
21949    False
21950    False
21951    False
21952    False
Length: 21953, dtype: bool

In [6]:
df.describe(include='all') # summary statistics

,lat_deg,lon_deg,status_id,report_date,source,water_source_clean,water_tech_clean,clean_country_id,clean_country_name,clean_adm1,...,is_urban,days_since_report,staleness,prediction_yes_0y,prediction_yes_2y,prediction_no_0y,prediction_no_2y,predicted_status_0y,predicted_status_2y,predicted_category
count,21953.000000,21953.000000,21953,21953,21953,21825,16034,21953,21953,21953,...,21953,21953.000000,21953.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
unique,NaN,NaN,3,736,15,10,15,1,1,39,...,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,NaN,Yes,2021-02-03,Evidence Action,Protected Well,Motorized Pump - Electric,KEN,Kenya,Kakamega,...,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,NaN,19761,783,10850,11693,6836,21953,21953,4470,...,21515,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,-0.125013,35.046013,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,2141.522024,56.929676,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,0.858474,1.117942,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,1630.550960,22.794620,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,-3.944933,33.952213,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,371.000000,17.921434,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,-0.535283,34.390438,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,752.000000,28.918776,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,0.142017,34.657778,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,1766.000000,57.311637,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,0.361275,34.918716,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,3936.000000,78.895948,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 3. Data Preparation 
We perform "Full Cleaning" here: fixing erroneous years, calculating infrastructure age, handling missing values, and ensuring the coordinates are valid.

In [7]:
# 3.1 Handling 'Install Year' & 'Water Point Age'
# Years before 1960 or in the future are statistically impossible for this tech.
df['install_year'] = pd.to_numeric(df['install_year'], errors='coerce')
df.loc[(df['install_year'] < 1960) | (df['install_year'] > 2024), 'install_year'] = np.nan

# Median Imputation: Robust against outliers
# Hierarchical imputation of install_year
df['install_year'] = df.groupby(['clean_adm1', 'water_tech_clean'])['install_year'].transform(
    lambda x: x.fillna(x.median()))
df['install_year'] = df.groupby('clean_adm1')['install_year'].transform(
    lambda x: x.fillna(x.median()))
df['install_year'] = df['install_year'].fillna(df['install_year'].median())

In [8]:
df.isna().sum() # checking if changes have taken place and install year has no missing values

lat_deg                        0
lon_deg                        0
status_id                      0
report_date                    0
source                         0
water_source_clean           128
water_tech_clean            5919
clean_country_id               0
clean_country_name             0
clean_adm1                     0
clean_adm2                     0
clean_adm3                     0
clean_adm4                 21953
activity_id                 2209
scheme_id                  21953
wpdx_id                        0
install_year                   0
installer                  18110
rehab_year                 21953
rehabilitator              21953
management_clean           11890
pay_clean                  18360
status_clean                   0
fecal_coliform_presence    21953
fecal_coliform_value       21953
subjective_quality         11719
notes                      18614
local_population             875
assigned_population          875
facility_type                  0
water_sour

Calculating water point age and clipping it at zero transforms a static installation year into a dynamic proxy for mechanical wear-and-tear while sanitizing logical data-entry errors to ensure statistical validity.

In [9]:
# Calculating Age based on report date
df['report_date'] = pd.to_datetime(df['report_date'], errors='coerce')
df['water_point_age'] = df['report_date'].dt.year - df['install_year']
df['water_point_age'] = df['water_point_age'].clip(lower=0)

In [10]:
# 3.2 Target Cleaning: Binary Mapping
# 0 = Functional, 1 = Non-Functional (Risk)
df['target'] = df['status_clean'].map({
    'Functional': 0, 'Functional, needs repair': 0, 'Functional, not in use': 0,
    'Non-Functional': 1, 'Non-Functional, dry season': 1, 'Abandoned/Decommissioned': 1
})
df = df.dropna(subset=['target'])

In [11]:
# 3.3 Feature Selection & Null Handling
# We selected features that impact physical wear and management effectiveness.
features = ['lat_deg', 'lon_deg', 'water_source_clean', 'water_tech_clean', 
            'management_clean', 'pay_clean', 'is_urban', 'water_point_age']

X = df[features].copy()
cat_cols = ['water_source_clean', 'water_tech_clean', 'management_clean', 'pay_clean', 'is_urban']
X[cat_cols] = X[cat_cols].fillna('Unknown')
y = df['target']

In [12]:
X.head() # Checking the selection

,lat_deg,lon_deg,water_source_clean,water_tech_clean,management_clean,pay_clean,is_urban,water_point_age
0,0.212005,34.615833,Protected Well,Motorized Pump - Electric,Unknown,Unknown,False,27.0
1,0.212567,34.616543,Protected Well,Motorized Pump - Electric,Unknown,Unknown,False,27.0
2,0.222063,34.601463,Protected Well,Motorized Pump - Electric,Unknown,Unknown,False,27.0
3,0.222362,34.604015,Protected Well,Motorized Pump - Electric,Unknown,Unknown,False,27.0
4,0.023602,34.777866,Protected Well,Motorized Pump - Electric,Unknown,Unknown,False,9.0


In [13]:
X.isna().sum() # selection is clean and has no missing values

lat_deg               0
lon_deg               0
water_source_clean    0
water_tech_clean      0
management_clean      0
pay_clean             0
is_urban              0
water_point_age       0
dtype: int64

In [14]:
# 3.4 Removing Duplicates
# Ensures the model doesn't overfit on repeated data entries.
df = df.drop_duplicates()